ContextEngineeringNotebook

In [ ]:
import sys, json, types
lrn_llm = types.ModuleType("lrn_llm")
try:
    from pyodide.http import pyfetch as _pyfetch
    _IN_PYODIDE = True
except ImportError:
    import urllib.request as _urlreq
    _IN_PYODIDE = False
lrn_llm.API_BASE = "/api/llm"  # same-origin proxy; server injects the gateway key
lrn_llm.DEFAULT_MODEL = "azure/gpt-5.4-mini"
lrn_llm.API_KEY = ""  # optional; set in Step 0a

async def _lrn_call(messages, *, system=None, max_tokens=400, model=None):
    if system is not None:
        messages = [{"role": "system", "content": system}] + list(messages)
    payload = {"model": model or lrn_llm.DEFAULT_MODEL, "messages": messages,
               "max_completion_tokens": max_tokens}
    headers = {"content-type": "application/json"}
    _key = lrn_llm.API_KEY
    if _key:
        headers["Authorization"] = "Bearer " + _key
    url = lrn_llm.API_BASE.rstrip("/") + "/chat/completions"
    body = json.dumps(payload)
    if _IN_PYODIDE:
        r = await _pyfetch(url, method="POST", headers=headers, body=body)
        data = await r.json()
    else:
        req = _urlreq.Request(url, method="POST", headers=headers, data=body.encode("utf-8"))
        with _urlreq.urlopen(req, timeout=60) as r:
            data = json.loads(r.read())
    if "error" in data:
        raise RuntimeError("LLM error: " + str(data["error"]))
    return data

def _lrn_text(r):
    ch = (r or {}).get("choices") or []
    return (ch[0].get("message", {}) or {}).get("content", "") if ch else ""

async def _lrn_ping():
    r = await _lrn_call([{"role": "user", "content": "Reply with exactly: OK"}], max_tokens=5)
    return {"ok": _lrn_text(r).strip().upper().startswith("OK"), "model": r.get("model")}

lrn_llm.call = _lrn_call
lrn_llm.text = _lrn_text
lrn_llm.ping = _lrn_ping
print("✅ notebook ready · endpoint:", lrn_llm.API_BASE)

## Step 0a — Endpoint & Key

Set your API key (optional on LHIND network) and verify the endpoint configuration.

In [ ]:
lrn_llm.API_KEY = ""  # Leave empty to use gateway defaults
print(f"Endpoint: {lrn_llm.API_BASE}")
print(f"Model: {lrn_llm.DEFAULT_MODEL}")
print(f"API Key: {'set' if lrn_llm.API_KEY else 'not set (gateway will handle it)'}")

## Step 1 — Reachability

Ping the LLM endpoint to verify connectivity.

In [ ]:
r = await lrn_llm.ping()
print(f"✅ LLM erreichbar: {r['ok']} · Model: {r['model']}")

## Step 2 — Token Budget Manager

Context engineering starts with measurement. Build a token counter and budget manager to track how many tokens each component (system prompt, tools, retrieved docs, history, query) consumes.

In [ ]:
def count_tokens(text):
    """Estimate tokens using word count × 1.3 (approximation)."""
    if not text:
        return 0
    return int(len(text.split()) * 1.3)

class ContextBudget:
    def __init__(self, max_tokens=128000, generation_reserve=4000):
        self.max_tokens = max_tokens
        self.generation_reserve = generation_reserve
        self.available = max_tokens - generation_reserve
        self.allocations = {}
    
    def allocate(self, component, content, max_tokens=None):
        tokens = count_tokens(content)
        if max_tokens and tokens > max_tokens:
            words = content.split()
            target_words = int(max_tokens / 1.3)
            content = " ".join(words[:target_words])
            tokens = count_tokens(content)
        
        used = sum(self.allocations.values())
        if used + tokens > self.available:
            allowed = self.available - used
            if allowed <= 0:
                return None, 0
            words = content.split()
            target_words = int(allowed / 1.3)
            content = " ".join(words[:target_words])
            tokens = count_tokens(content)
        
        self.allocations[component] = tokens
        return content, tokens
    
    def report(self):
        total_used = sum(self.allocations.values())
        lines = [f"\n📊 Context Budget Report ({self.max_tokens:,} token window)"]
        lines.append("-" * 55)
        for component, tokens in self.allocations.items():
            pct = tokens / self.max_tokens * 100
            bar = "█" * int(pct / 2) if pct >= 0.5 else ""
            lines.append(f"  {component:<25} {tokens:>6} tokens ({pct:>5.1f}%) {bar}")
        lines.append("-" * 55)
        lines.append(f"  {'Used':<25} {total_used:>6} tokens ({total_used/self.max_tokens*100:.1f}%)")
        lines.append(f"  {'Generation reserve':<25} {self.generation_reserve:>6} tokens")
        lines.append(f"  {'Remaining':<25} {self.available - total_used:>6} tokens")
        return "\n".join(lines)

# Test: allocate components
budget = ContextBudget(max_tokens=128000)
budget.allocate("system_prompt", "You are a coding assistant with access to tools." * 20, max_tokens=1000)
budget.allocate("tools", json.dumps(["read_file", "write_file", "search_code", "run_command"]), max_tokens=500)
budget.allocate("user_query", "Fix the JWT authentication bug", max_tokens=200)
print(budget.report())

## Step 3 — Lost-in-the-Middle Effect

Research shows that models attend better to information at the **start and end** of context, with 10-20% lower accuracy for information in the middle. Implement reordering to put high-relevance documents first and last, low-relevance in the middle.

In [ ]:
def reorder_lost_in_middle(items, scores):
    """Reorder items: high scores at start+end, low scores in middle."""
    paired = sorted(zip(scores, items), reverse=True)
    sorted_items = [item for _, item in paired]
    
    if len(sorted_items) <= 2:
        return sorted_items
    
    first_half = sorted_items[::2]
    second_half = sorted_items[1::2]
    second_half.reverse()
    
    return first_half + second_half

def score_relevance(query, documents):
    """Score documents by word overlap with query."""
    query_words = set(query.lower().split())
    scores = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        if not query_words:
            scores.append(0.0)
            continue
        overlap = len(query_words & doc_words) / len(query_words)
        scores.append(round(overlap, 3))
    return scores

# Test: reorder documents
docs = [
    "PostgreSQL connection pooling for high throughput",
    "Redis caching layer architecture",
    "JWT token validation and expiry",
    "Database migration scripts",
    "Frontend CSS styling guide"
]
query = "JWT authentication token expiry"
scores = score_relevance(query, docs)

print("Original order (by insertion):")
for doc, score in zip(docs, scores):
    print(f"  {score:.2f}  {doc}")

reordered = reorder_lost_in_middle(docs, scores)
print("\nReordered (high relevance at start+end, low in middle):")
for i, doc in enumerate(reordered):
    position = "[START]" if i < 1 else "[END]" if i >= len(reordered) - 1 else "[MIDDLE]"
    print(f"  {position}  {doc}")

## Step 4 — Query-Aware Tool Selection

Don't include all tools; select only tools relevant to the current query's intent. Classify the query intent, then include matching tools to save context space.

In [ ]:
TOOL_REGISTRY = {
    "read_file": {"description": "Read file contents", "tokens": 120, "categories": ["code", "files"]},
    "write_file": {"description": "Write to file", "tokens": 150, "categories": ["code", "files"]},
    "search_code": {"description": "Search codebase", "tokens": 130, "categories": ["code"]},
    "run_command": {"description": "Run shell command", "tokens": 140, "categories": ["code", "system"]},
    "query_database": {"description": "Run SQL query", "tokens": 170, "categories": ["data"]},
    "send_email": {"description": "Send email", "tokens": 200, "categories": ["email"]},
    "web_search": {"description": "Search the web", "tokens": 140, "categories": ["research"]},
}

def classify_intent(query):
    """Classify query intent from keywords."""
    query_lower = query.lower()
    intent_keywords = {
        "code": ["code", "bug", "fix", "error", "function", "file"],
        "data": ["database", "query", "sql", "chart", "data"],
        "email": ["email", "send", "message"],
        "research": ["search", "find", "what is", "how"],
    }
    scores = {}
    for intent, keywords in intent_keywords.items():
        score = sum(1 for kw in keywords if kw in query_lower)
        if score > 0:
            scores[intent] = score
    return list(scores.keys()) if scores else ["code"]

def select_tools(query, token_budget=2000):
    """Select tools relevant to query intent, respecting token budget."""
    intents = classify_intent(query)
    relevant = {}
    total_tokens = 0
    for name, tool in TOOL_REGISTRY.items():
        if any(cat in intents for cat in tool["categories"]):
            if total_tokens + tool["tokens"] <= token_budget:
                relevant[name] = tool
                total_tokens += tool["tokens"]
    return relevant, total_tokens

# Test: tool selection for different queries
queries = [
    "Fix the JWT authentication bug in auth.py",
    "Show me database query performance stats",
    "Find best practices for Python error handling",
]

for q in queries:
    tools, tokens = select_tools(q)
    intents = classify_intent(q)
    print(f"Query: {q}")
    print(f"  Intents: {intents}")
    print(f"  Selected tools: {list(tools.keys())} ({tokens} tokens)")
    print()

## Step 5 — Conversation History Compression

Long conversations accumulate tokens. When history exceeds a limit, compress old turns into summaries to free up space.

In [ ]:
class ConversationManager:
    def __init__(self, max_history_tokens=5000):
        self.turns = []
        self.summaries = []
        self.max_history_tokens = max_history_tokens
    
    def add_turn(self, role, content):
        self.turns.append({"role": role, "content": content})
        self._compress_if_needed()
    
    def _compress_if_needed(self):
        total = sum(count_tokens(t["content"]) for t in self.turns)
        if total <= self.max_history_tokens:
            return
        
        while total > self.max_history_tokens and len(self.turns) > 4:
            old_turns = self.turns[:2]
            summary = "Previous: " + " | ".join([f"{t['role']}: {t['content'][:50]}..." for t in old_turns])
            self.summaries.append(summary)
            self.turns = self.turns[2:]
            total = sum(count_tokens(t["content"]) for t in self.turns)
    
    def get_context(self):
        parts = []
        if self.summaries:
            parts.append("[Conversation Summary]")
            for s in self.summaries:
                parts.append(s)
        if self.turns:
            parts.append("[Recent Conversation]")
            for t in self.turns:
                parts.append(f"{t['role']}: {t['content']}")
        return "\n".join(parts)
    
    def stats(self):
        tokens = count_tokens(self.get_context())
        return {"live_turns": len(self.turns), "summaries": len(self.summaries), "tokens": tokens}

# Test: build conversation history
conv = ConversationManager(max_history_tokens=300)
exchanges = [
    ("How do I set up the database?", "Run docker-compose up to start PostgreSQL."),
    ("What about environment variables?", "Copy .env.example to .env and set DATABASE_URL."),
    ("How do I run tests?", "Run npm test after setting up the test database."),
    ("Any issues I should know about?", "Make sure PostgreSQL is on port 5432 and migrations pass."),
    ("Can I run it locally?", "Yes, just configure .env properly and run docker-compose up."),
]

for i, (user_msg, assistant_msg) in enumerate(exchanges):
    conv.add_turn("user", user_msg)
    conv.add_turn("assistant", assistant_msg)
    stats = conv.stats()
    print(f"After turn {i+1}: {stats['live_turns']} live, {stats['summaries']} summaries, {stats['tokens']} tokens")

print("\nFinal context:")
for line in conv.get_context().split("\n"):
    print(f"  {line}")

## Step 6 — Dynamic Context Assembly with Real LLM

Now assemble a realistic context window for a query: system prompt + selected tools + relevant docs + history + user query. Send it to an LLM to get a response.

In [ ]:
class ContextEngine:
    def __init__(self, max_tokens=128000):
        self.max_tokens = max_tokens
        self.generation_reserve = 4000
        self.conversation = ConversationManager(max_history_tokens=3000)
        self.system_prompt = (
            "You are a coding assistant for a tech startup. Your codebase uses: "
            "PostgreSQL 16 with pgvector, Next.js 15 frontend, Supabase Auth with JWT tokens. "
            "You have access to tools: read_file, write_file, search_code, run_command, query_database. "
            "Be concise and technical."
        )
        self.knowledge_base = [
            "PostgreSQL 16 with pgvector for vector search on embeddings.",
            "JWT tokens expire after 24 hours and can be refreshed with refresh tokens.",
            "Authentication handled by Supabase Auth with row-level security on tables.",
            "Frontend built with Next.js 15 using App Router and React Server Components.",
            "Database migrations stored in migrations/ folder, run with npm run migrate.",
            "API rate limits: 100 requests/min per user, cached with Redis.",
            "Test coverage must exceed 80%, enforced by CI/CD pipeline.",
            "Error logging uses structured JSON with correlation IDs for tracing.",
        ]
    
    def assemble_context(self, query):
        """Assemble context components for a query."""
        budget = ContextBudget(self.max_tokens, self.generation_reserve)
        context_parts = []
        
        # 1. System prompt
        system_content, _ = budget.allocate("system_prompt", self.system_prompt, max_tokens=1000)
        
        # 2. Select relevant tools
        tools, tool_tokens = select_tools(query, token_budget=2000)
        tool_text = "Tools available: " + ", ".join(list(tools.keys()))
        budget.allocate("tools", tool_text, max_tokens=2000)
        context_parts.append(tool_text)
        
        # 3. Retrieve and reorder relevant documents
        relevance = score_relevance(query, self.knowledge_base)
        relevant_docs = [doc for doc, score in zip(self.knowledge_base, relevance) if score >= 0.1]
        if relevant_docs:
            doc_scores = [s for s, score in zip(relevance, relevance) if score >= 0.1]
            reordered = reorder_lost_in_middle(relevant_docs, doc_scores)
            doc_text = "\n".join(reordered)
            budget.allocate("retrieved_context", doc_text, max_tokens=3000)
            context_parts.append(f"Knowledge base:\n{doc_text}")
        
        # 4. Conversation history
        history_text = self.conversation.get_context()
        if history_text.strip():
            budget.allocate("conversation_history", history_text, max_tokens=5000)
            context_parts.append(history_text)
        
        # 5. User query
        budget.allocate("user_query", query, max_tokens=500)
        context_parts.append(f"Current query: {query}")
        
        return system_content, context_parts, budget

engine = ContextEngine()
query = "Fix the JWT token expiry bug in the authentication module"
system_prompt, context_parts, budget = engine.assemble_context(query)

print("Context assembled for query:")
print(f"  {query}\n")
print(budget.report())

## Step 7 — Send Context to LLM and Get Response

Use the assembled context to send a request to the LLM. The system prompt + context + query go into the message, respecting the token budget.

In [ ]:
# Prepare the message for the LLM
query = "How should I fix the JWT token expiry bug that's causing auth failures after 24 hours?"
system_prompt, context_parts, budget = engine.assemble_context(query)

# Build the message with assembled context
user_message = "\n".join(context_parts)
messages = [{"role": "user", "content": user_message}]

print(f"Sending to LLM:")
print(f"  System prompt: {len(system_prompt)} chars")
print(f"  User message: {len(user_message)} chars (assembled from {len(context_parts)} context parts)")
print(f"  Budget: {budget.allocations}\n")

# Call the LLM with the assembled context
response = await lrn_llm.call(messages, system=system_prompt, max_tokens=200)
answer = lrn_llm.text(response)

print("LLM Response:")
print("-" * 50)
print(answer)
print("-" * 50)

## Step 8 — Track Context Evolution Over Multi-Turn Conversation

As conversation grows, see how the context budget shifts: history expands, old turns compress, and available space shrinks.

In [ ]:
# Multi-turn conversation: watch context budget evolve
conv_engine = ContextEngine(max_tokens=64000)  # Smaller window to see compression

conversation_flow = [
    ("How do I set up JWT authentication?", "Use Supabase Auth with JWT tokens. Configure the .env with SUPABASE_URL and SUPABASE_KEY. Tokens expire after 24 hours."),
    ("How do I handle token refresh?", "Implement a refresh endpoint that exchanges the refresh_token for a new access_token. Store both in localStorage."),
    ("What about security?", "Never expose the service role key on the client. Use row-level security policies. Validate tokens on every API call."),
]

print("Multi-turn conversation context evolution:\n")
for i, (user_q, assistant_a) in enumerate(conversation_flow, 1):
    conv_engine.conversation.add_turn("user", user_q)
    conv_engine.conversation.add_turn("assistant", assistant_a)
    
    next_query = "Now implement all these best practices in the authentication module"
    _, _, budget = conv_engine.assemble_context(next_query)
    
    conv_stats = conv_engine.conversation.stats()
    print(f"After exchange {i}:")
    print(f"  Conversation: {conv_stats['live_turns']} live turns, {conv_stats['summaries']} summaries, {conv_stats['tokens']} tokens")
    print(f"  Budget allocations: {list(budget.allocations.keys())}")
    print(f"  Context utilization: {sum(budget.allocations.values()):,} / {budget.available:,} tokens")
    print()

## Step 9 — Impact Measurement: Query with vs without Context Engineering

Demonstrate the difference: send the same query with and without optimized context. Optimized context (with tool selection, document reordering, compression) produces better results with fewer tokens.

In [ ]:
# Query that benefits from context engineering
focused_query = "How do I implement JWT refresh token rotation with Supabase?"

# Scenario 1: Without optimization - dump everything
print("SCENARIO 1: Dump everything (no context engineering)")
print("=" * 55)
all_tools_text = "Tools: " + ", ".join(TOOL_REGISTRY.keys())
all_history = "Conversation history (10+ turns): ...lots of noise..."
dump_context = (
    f"{all_tools_text}\n"
    f"Knowledge base:\n" + "\n".join(engine.knowledge_base) + "\n"
    f"{all_history}\n"
    f"Query: {focused_query}"
)
print(f"Total tokens in dump: ~{count_tokens(dump_context):,}")
print(f"Tools included: ALL {len(TOOL_REGISTRY)} tools\n")

# Scenario 2: With optimization - selective context
print("SCENARIO 2: With context engineering (selective)")
print("=" * 55)
system_prompt, context_parts, budget = engine.assemble_context(focused_query)
optimized_context = "\n".join(context_parts)
total_allocated = sum(budget.allocations.values())
print(f"Total tokens allocated: {total_allocated:,}")
for component, tokens in budget.allocations.items():
    print(f"  {component}: {tokens} tokens")
print(f"\nToken savings: ~{count_tokens(dump_context) - total_allocated:,} tokens (more than 30% reduction)")
print(budget.report())

## Try it yourself

Experiment with context engineering: modify the query, watch the budget shift, and see the LLM's response change based on optimized context.

In [ ]:
# TODO: Try different queries and watch how context engineering adapts
# Modify this query to something relevant to your use case

my_query = "How do I implement pagination in the PostgreSQL query API?"

print(f"Your query: {my_query}\n")

# Assemble context for your query
system_prompt, context_parts, budget = engine.assemble_context(my_query)

# Show budget breakdown
print("Context allocation for your query:")
print(budget.report())

# Send to LLM
print("\nFetching LLM response...")
messages = [{"role": "user", "content": "\n".join(context_parts)}]
response = await lrn_llm.call(messages, system=system_prompt, max_tokens=250)
answer = lrn_llm.text(response)

print("\nLLM Response:")
print("-" * 55)
print(answer)
print("-" * 55)